This is the notebook where i am trying to learn how can i build a tiny GPT-style character level language model from scratch in pytorch

In [3]:
import torch # PyTorch library for deep learning

with open("input.txt", "r") as file:
    text = file.read()

# print(text)
print("length:",len(text))

ModuleNotFoundError: No module named 'torch'

In [3]:
chars = sorted(list(set(text)))
print("vocabulary size:",len(chars))

vocabulary size: 21


we cant use directly characters because model uses integers so we need to convert every character to integer that called encoding and decoding 
for that we will create two dictionaries stoi(string to integer) and itos(integer to string)

In [5]:
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}

In [ ]:
# encode/decode funtions 
def encode(s):
    return [stoi[c] for c in s]

def decode(ids):
    return ''.join(itos[i] for i in ids)

# print(decode(encode("hello")))

hello


now lets convert the entire data to a tensor 

In [8]:
data = torch.tensor(encode(text),dtype=torch.long)
print(data)

tensor([ 8,  5, 10, 10, 13,  1, 19, 13, 14, 10,  4,  0,  8,  5, 10, 10, 13,  1,
        16,  8,  5, 14,  5,  0,  8, 13, 19,  1,  2, 14,  5,  1, 20, 13, 17,  0,
         9,  1,  2, 11,  1, 10,  5,  2, 14, 12,  9, 12,  7,  1, 16, 14,  2, 12,
        15,  6, 13, 14, 11,  5, 14, 15,  0, 16, 14,  2, 12, 15,  6, 13, 14, 11,
         5, 14, 15,  1,  2, 14,  5,  1,  9, 12, 16,  5, 14,  5, 15, 16,  9, 12,
         7,  0,  9,  1, 10, 13, 18,  5,  1, 11,  2,  3,  8,  9, 12,  5,  1, 10,
         5,  2, 14, 12,  9, 12,  7])


lets split the data into train and test data set

In [9]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

instead to taking one character at a time we have to train it in chunks 

In [ ]:
block_size = 8

In [11]:
#create a training batch 
torch.manual_seed(1337)
batch_size = 4
block_size = 8
def get_batch(split):
    data_source = train_data if split =="train" else val_data

    ix = torch.randint(
        len(data_source)-block_size,(batch_size,)
    )

    x = torch.stack([
        data_source[i:i+block_size] for i in ix
    ])

    y = torch.stack([
        data_source[i + 1:i + block_size + 1]
        for i in ix
    ])

    return x, y
    

now lets get the train data that we will use for training our first model

In [12]:
xb, yb = get_batch("train")#xb input batch and yb is output means the answer we are expecting from the model
#Input:
# h e l l o _ w o

# Target:
# e l l o _ w o r
# here we can see that target is shifted by one position
print(xb)
print(yb)
print(xb.shape)
print(yb.shape)

tensor([[ 5,  1, 20, 13, 17,  0,  9,  1],
        [ 1, 16,  8,  5, 14,  5,  0,  8],
        [12,  7,  1, 16, 14,  2, 12, 15],
        [ 5,  1, 20, 13, 17,  0,  9,  1]])
tensor([[ 1, 20, 13, 17,  0,  9,  1,  2],
        [16,  8,  5, 14,  5,  0,  8, 13],
        [ 7,  1, 16, 14,  2, 12, 15,  6],
        [ 1, 20, 13, 17,  0,  9,  1,  2]])
torch.Size([4, 8])
torch.Size([4, 8])


the embedding layer we need to add right now we have (B,T) but we want (B,T,C) vectors 

In [13]:
import torch
import torch.nn as nn

# Example
vocab_size = 21
C = 32

token_embedding_table = nn.Embedding(vocab_size, C)

x = token_embedding_table(xb)

print(x.shape)

torch.Size([4, 8, 32])


now we get learnable lookup table 
Suppose initially:
    'h' → [0.1, 0.8, -0.2, ...]
During training, the model makes mistakes.

The loss tells us:

"These parameters need to change."

Gradient descent then updates the embedding vector.

After a lot of training:
    'h' → [0.42, 0.17, -0.81, ...]

The model has learned useful representations.

This is why you shouldn't think of embeddings as fixed definitions.

They are parameters learned by the model.

We've now represented:

h e l l o

as vectors.

But consider:

h e l l o

and:

o l l e h

The same characters are present.

The embedding of h is identical in both cases.

The embedding layer itself doesn't know:

"This h is the first character."

It only knows:

h → vector H

It doesn't know where h occurs in the sequence.

That's a problem.